# Group Shared Hashtags Example

Load the saved Step-3 graph and post data for one campaign, select the top author's connected k-core group, and create the shared-hashtag example.

In [ ]:
# Campaign directory name, for example: Iran_1.
campaign_name = "Iran_1"

time_window = "1h"


In [ ]:
from pathlib import Path

import pandas as pd


def find_project_root(start_path: Path) -> Path:
    """Find the project root containing the source and results folders."""
    for candidate in (start_path.resolve(), *start_path.resolve().parents):
        if (candidate / "src").is_dir() and (candidate / "results").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the Backbone-Graph project root.")


project_root = find_project_root(Path.cwd())
topic_col = "BERTopic_topic_256"
top_percentage = 100
dataset_part = f"{campaign_name}_part_all"
step_3_path = (
    project_root / "results" / "Labeled_Datasets" / campaign_name
    / dataset_part / "step_3_key_authors_graph"
    / f"topic_col_{topic_col}_top_percentage_{top_percentage:g}"
)
graph_stem = f"accounts_graph_topic_col_{topic_col}"
graph_path = step_3_path / "account_graphs" / f"{graph_stem}.graphml"
input_data_path = step_3_path / f"{dataset_part}_step_3.gzip.parquet"
window_offset = pd.tseries.frequencies.to_offset(time_window)
if window_offset.nanos <= 0:
    raise ValueError("time_window must be a positive fixed duration.")
time_window = window_offset.freqstr
output_path = (
    step_3_path / "top_k_core_authors" / "shared_hashtags_example"
    / f"time_delta_{time_window}"
)

missing_paths = [path for path in [graph_path, input_data_path] if not path.is_file()]
if missing_paths:
    missing_text = "\n".join(f"- {path}" for path in missing_paths)
    raise FileNotFoundError(f"Missing Step-3 input files:\n{missing_text}")

print(f"Step-3 graph: {graph_path}")
print(f"Step-3 post data: {input_data_path}")
print(f"Output folder: {output_path}")


In [ ]:
import json
import re

import matplotlib.pyplot as plt
import nbformat
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display


def write_report(message: str) -> None:
    """Provide the source notebook's reporting hook without file output."""
    return None


def load_notebook_function(notebook_path: Path, function_name: str) -> None:
    """Load one function definition from a Jupyter notebook.

    Args:
        notebook_path: Notebook containing the function definition.
        function_name: Name of the function to load into this notebook.
    """
    notebook = nbformat.read(notebook_path, as_version=4)
    marker = f"def {function_name}("
    source = next(
        (cell.source for cell in notebook.cells
         if cell.cell_type == "code" and marker in cell.source),
        None,
    )
    if source is None:
        raise ValueError(f"Could not find {function_name} in {notebook_path}.")

    exec(source.replace("@report_done\n", ""), globals())


source_notebook_path = project_root / "src" / "3_key_authors_graph.ipynb"
load_notebook_function(source_notebook_path, "get_node_k_core")
load_notebook_function(source_notebook_path, "get_top_authors_by_indicator")
load_notebook_function(source_notebook_path, "get_io_authors_by_ratio")
load_notebook_function(source_notebook_path, "create_group_shared_hashtags_example")


In [ ]:
account_id_col = "accountid"
post_time_col = "post_time"
required_columns = [
    account_id_col, post_time_col, "hashtags", "translated_post",
    "post_text", "postid", "is_repost", "reposted_postid", "is_control",
]

G_accounts = nx.read_graphml(graph_path)
if G_accounts.is_directed():
    G_accounts = G_accounts.to_undirected()

core_numbers = nx.core_number(G_accounts)
node_indicators_df = pd.DataFrame.from_dict(
    core_numbers,
    orient="index",
    columns=["core_number"],
)
top_core_user = get_top_authors_by_indicator(
    node_indicators_df,
    indicator_col="core_number",
    n=10,
).index[0]
group_accountids = sorted(
    get_node_k_core(
        node_id=top_core_user,
        G=G_accounts,
        core_numbers=core_numbers,
        include_node=True,
    ),
    key=str,
)

df = pd.read_parquet(input_data_path, columns=required_columns)
print(f"Graph: {G_accounts.number_of_nodes():,} nodes, {G_accounts.number_of_edges():,} edges")
print(f"Loaded {len(df):,} posts.")
print(f"Top core author: {top_core_user} (core number {core_numbers[top_core_user]:,})")
print(f"Group accounts: {len(group_accountids):,}")


In [ ]:
group_shared_hashtag_posts_df = create_group_shared_hashtags_example(
    df=df,
    nodes=set(group_accountids),
    time_window=time_window,
    output_path=output_path,
)

group_shared_hashtag_posts_df


In [ ]:
def save_latex_table(dataframe: pd.DataFrame, output_file: Path) -> None:
    """Save a DataFrame as a standalone LaTeX table.

    Args:
        dataframe: Table to export.
        output_file: Destination .tex file.
    """
    output_file.write_text(
        dataframe.to_latex(index=False, escape=True),
        encoding="utf-8",
    )


window_summary_df = pd.read_csv(output_path / "shared_hashtag_window_summary.csv")
group_accountids_df = pd.read_csv(output_path / "group_accountids.csv")
top_words_df = pd.read_csv(output_path / "peak_shared_hashtag_top_50_words.csv")
hashtag_counts_df = pd.read_csv(output_path / "peak_shared_hashtag_hashtag_counts.csv")
word_counts_df = pd.read_csv(output_path / "peak_shared_hashtag_word_counts.csv")
display(hashtag_counts_df.head(50), word_counts_df.head(50))
save_latex_table(hashtag_counts_df, output_path / "peak_shared_hashtag_hashtag_counts.tex")
save_latex_table(word_counts_df, output_path / "peak_shared_hashtag_word_counts.tex")

save_latex_table(group_accountids_df, output_path / "group_accountids.tex")
save_latex_table(window_summary_df, output_path / "shared_hashtag_window_summary.tex")
save_latex_table(top_words_df, output_path / "peak_shared_hashtag_top_50_words.tex")
save_latex_table(
    group_shared_hashtag_posts_df,
    output_path / "peak_shared_hashtag_posts.tex",
)

print(f"Saved CSV, LaTeX, and available word-cloud image outputs to: {output_path}")
